# Generate Synthetic CS2 Commentary Training Data

Use this notebook to generate synthetic training rows for a Counter-Strike 2 commentary model.

The notebook expects seed examples shaped like the current v3 wrapper output:

```json
{
  "input": {
    "context": {
      "score": {...},
      "alive_players": [...]
    },
    "previous_events": [...],
    "current_events": [...],
    "request": {
      "mode": "event_bundle",
      "lines": [...]
    }
  }
}
```

It generates synthetic rows shaped like:

```json
{
  "input": {...},
  "output": {
    "commentary": "line 1\nline 2"
  }
}
```

This notebook is specifically for **CS2**. The only caster labels used are `play_by_play` and `color`.


## Data generation settings


In [1]:
NUM_TRAIN_EXAMPLES = 5  # @param {type:"number"}
NUM_VAL_EXAMPLES = 1  # @param {type:"number"}
NUM_TEST_EXAMPLES = 1  # @param {type:"number"}
TEMPERATURE = 0.7  # @param {type:"number"}
MAX_ROW_RETRIES = 5  # @param {type:"number"}

DATA_FOLDER = "./.data/generated_cs2"
SEED_EXAMPLES_FILE = "./training_wrapper_pretty.jsonl"  # upload or mount this in Colab
MAX_SEED_EXAMPLES = 50  # @param {type:"number"}
SEED_EXAMPLES_PER_PROMPT = 3  # @param {type:"number"}

DATAGEN_URL = "https://openrouter.ai/api/v1"
DATAGEN_MODEL = "openai/gpt-5.4-nano"

ALLOWED_CASTERS = ["play_by_play", "color"]
ALLOWED_PROMPT_STYLES = ["play_by_play_event", "play_by_play_follow_up", "idle_color"]
ALLOWED_REQUEST_MODES = ["event_bundle", "idle_color", "idle_conversation"]

INLINE_SEED_EXAMPLES = []

from pathlib import Path
Path(DATA_FOLDER).mkdir(parents=True, exist_ok=True)


## Seed examples and schema


In [2]:
import json
from pathlib import Path

def load_seed_examples(seed_file: str, inline_examples: list[dict], max_examples: int) -> list[dict]:
    examples: list[dict] = []
    path = Path(seed_file)

    if path.exists():
        text = path.read_text(encoding="utf-8")

        chunks = [chunk.strip() for chunk in text.split("\n\n") if chunk.strip()]
        if chunks:
            for chunk in chunks:
                record = json.loads(chunk)
                if isinstance(record, dict) and isinstance(record.get("input"), dict):
                    examples.append(record)
        else:
            for raw_line in text.splitlines():
                line = raw_line.strip()
                if not line:
                    continue
                record = json.loads(line)
                if isinstance(record, dict) and isinstance(record.get("input"), dict):
                    examples.append(record)

    for record in inline_examples:
        if isinstance(record, dict) and isinstance(record.get("input"), dict):
            examples.append(record)

    if max_examples > 0:
        examples = examples[:max_examples]

    return examples


SEED_EXAMPLES = load_seed_examples(SEED_EXAMPLES_FILE, INLINE_SEED_EXAMPLES, MAX_SEED_EXAMPLES)
print(f"Loaded {len(SEED_EXAMPLES)} seed examples")
if SEED_EXAMPLES:
    print(json.dumps(SEED_EXAMPLES[0], indent=2, sort_keys=True))


## Model for structured output


In [3]:
from typing import Any, Literal
from pydantic import BaseModel, Field


class RequestLine(BaseModel):
    caster: Literal["play_by_play", "color"]
    style: Literal["play_by_play_event", "play_by_play_follow_up", "idle_color"]


class RequestSpec(BaseModel):
    mode: Literal["event_bundle", "idle_color", "idle_conversation"]
    lines: list[RequestLine] = Field(default_factory=list)


class SyntheticInput(BaseModel):
    context: dict[str, Any]
    previous_events: list[dict[str, Any]] = Field(default_factory=list)
    current_events: list[dict[str, Any]] = Field(default_factory=list)
    request: RequestSpec


class SyntheticOutput(BaseModel):
    commentary: str


class SyntheticTrainingRow(BaseModel):
    input: SyntheticInput
    output: SyntheticOutput


## Get OpenRouter API key


In [4]:
import sys
import os
from dotenv import load_dotenv

if 'google.colab' in sys.modules:
  from google.colab import userdata # type:ignore
  os.environ['OPENROUTER_API_KEY'] = userdata.get('OPENROUTER_API_KEY')
else:
  load_dotenv()

## Synthetic generation functions


In [5]:
import json
import openai
import os
import random

client = openai.OpenAI(
    base_url=DATAGEN_URL,
    api_key=os.environ.get("OPENROUTER_API_KEY"),
)


FEW_SHOT_EXAMPLES = [
    {
        "input": {
            "context": {
                "score": {"CT": 2, "T": 5},
                "alive_players": [
                    {"name": "Colin", "team": "CT", "map_callout": "B Doors"},
                    {"name": "Maru", "team": "T", "map_callout": "B Site"}
                ]
            },
            "previous_events": [],
            "current_events": [
                {
                    "event_type": "kill",
                    "killer": {"name": "Colin", "team": "CT", "map_callout": "B Doors"},
                    "victim": {"name": "Maru", "team": "T"}
                }
            ],
            "request": {
                "mode": "event_bundle",
                "lines": [
                    {"caster": "play_by_play", "style": "play_by_play_event"},
                    {"caster": "color", "style": "play_by_play_follow_up"}
                ]
            }
        },
        "output": {"commentary": "Colin drops Maru.\nB is cracked open."}
    },
    {
        "input": {
            "context": {
                "score": {"CT": 4, "T": 4},
                "alive_players": [
                    {"name": "Niles", "team": "T", "map_callout": "Top Mid"}
                ]
            },
            "previous_events": [],
            "current_events": [
                {
                    "event_type": "grenade_detonated",
                    "grenade_type": "smoke",
                    "detonation_callout": "Top Mid",
                    "owner_player": {"name": "Niles", "team": "T", "map_callout": "Top Mid"}
                }
            ],
            "request": {
                "mode": "event_bundle",
                "lines": [
                    {"caster": "play_by_play", "style": "play_by_play_event"},
                    {"caster": "color", "style": "play_by_play_follow_up"}
                ]
            }
        },
        "output": {"commentary": "Smoke blooms Top Mid.\nThat shuts the peek down."}
    },
    {
        "input": {
            "context": {
                "score": {"CT": 6, "T": 5},
                "alive_players": [
                    {"name": "Yanni", "team": "CT", "map_callout": "A Site"},
                    {"name": "Tony", "team": "T", "map_callout": "Long"}
                ]
            },
            "previous_events": [
                {"event_type": "bomb_event", "state_after": "planted"}
            ],
            "current_events": [],
            "request": {
                "mode": "idle_color",
                "lines": [
                    {"caster": "color", "style": "idle_color"},
                    {"caster": "color", "style": "idle_color"},
                    {"caster": "color", "style": "idle_color"}
                ]
            }
        },
        "output": {"commentary": "This feels tense.\nNobody wants the first mistake.\nA still looks fragile."}
    },
    {
        "input": {
            "context": {
                "score": {"CT": 7, "T": 6},
                "alive_players": [
                    {"name": "Walt", "team": "CT", "map_callout": "Top Mid"},
                    {"name": "Uri", "team": "T", "map_callout": "Xbox"}
                ]
            },
            "previous_events": [
                {"event_type": "grenade_detonated", "grenade_type": "smoke", "detonation_callout": "Xbox"}
            ],
            "current_events": [],
            "request": {
                "mode": "idle_conversation",
                "lines": [
                    {"caster": "play_by_play", "style": "idle_color"},
                    {"caster": "color", "style": "idle_color"},
                    {"caster": "play_by_play", "style": "idle_color"}
                ]
            }
        },
        "output": {"commentary": "CT still own Mid.\nThat smoke bought real time.\nT need a clean next move."}
    }
]


def build_seed_block(seed_examples: list[dict]) -> str:
    if not seed_examples:
        return "[]"
    return json.dumps(seed_examples, indent=2, sort_keys=True)


def build_few_shot_block() -> str:
    return json.dumps(FEW_SHOT_EXAMPLES, indent=2, sort_keys=True)


def choose_request() -> dict:
    mode = random.choice(ALLOWED_REQUEST_MODES)
    if mode == "event_bundle":
        return {
            "mode": "event_bundle",
            "lines": [
                {"caster": "play_by_play", "style": "play_by_play_event"},
                {"caster": "color", "style": "play_by_play_follow_up"}
            ],
        }
    if mode == "idle_conversation":
        return {
            "mode": "idle_conversation",
            "lines": [
                {"caster": "play_by_play", "style": "idle_color"},
                {"caster": "color", "style": "idle_color"},
                {"caster": "play_by_play", "style": "idle_color"}
            ],
        }
    return {
        "mode": "idle_color",
        "lines": [
            {"caster": "color", "style": "idle_color"},
            {"caster": "color", "style": "idle_color"},
            {"caster": "color", "style": "idle_color"}
        ],
    }


def extract_json_object(raw_text: str) -> dict:
    raw_text = raw_text.strip()
    if raw_text.startswith("```"):
        raw_text = raw_text.strip("`")
        if raw_text.startswith("json"):
            raw_text = raw_text[4:]
        raw_text = raw_text.strip()
    try:
        return json.loads(raw_text)
    except json.JSONDecodeError:
        start = raw_text.find("{")
        end = raw_text.rfind("}")
        if start == -1 or end == -1 or end <= start:
            raise ValueError(f"Model did not return a valid JSON object: {raw_text[:400]}")
        candidate = raw_text[start:end+1]
        try:
            return json.loads(candidate)
        except json.JSONDecodeError as error:
            raise ValueError(f"Model returned invalid JSON: {candidate[:400]}") from error


def generate_synthetic_row(seed_examples: list[dict]) -> SyntheticTrainingRow:
    if not seed_examples:
        raise ValueError("No seed examples loaded. Upload or point SEED_EXAMPLES_FILE to v3 training wrapper JSONL.")

    sampled_examples = random.sample(seed_examples, k=min(SEED_EXAMPLES_PER_PROMPT, len(seed_examples)))
    request = choose_request()
    expected_line_count = len(request["lines"])

    prompt = f'''
You are generating synthetic supervised fine-tuning data for a Counter-Strike 2 commentary model.

Important context:
- This is CS2, not generic esports.
- The only caster labels are: play_by_play, color.
- Do not mention Portal, Turret, Announcer, or any character personalities.
- Keep commentary grounded in current_events when current_events is non-empty.
- previous_events is light context only and must never override current_events.
- context.score and context.alive_players are the only compact global context provided.
- The generated row must stay plausible for Counter-Strike 2.
- output.commentary must contain exactly one line per input.request.lines entry.
- Separate commentary lines with literal newline characters.

Line rules:
- play_by_play_event: ideally 2 to 6 words, never exceed 8 words.
- play_by_play_follow_up: ideally 3 to 8 words, never exceed 10 words.
- idle_color: ideally 4 to 10 words, never exceed 12 words.
- In event_bundle mode, line 1 is the trigger call and later lines are follow-up or color lines.
- In idle_conversation mode, the lines should feel like a brief back-and-forth.
- In idle_color mode, the lines should feel like one caster continuing the thought naturally.

Style guidance:
- play_by_play is direct, clipped, factual.
- color is dry, observant, understated.
- Prefer exact player names and callouts when present.
- Prefer literal, speakable phrasing over dramatic phrasing.
- Avoid generic filler like: locks it down, makes a play, finds a frag, huge kill, big round, what a play, comes up big.

Few-shot examples:
{build_few_shot_block()}

Seed examples for structure and event diversity:
{build_seed_block(sampled_examples)}

Generate exactly one new synthetic row with this JSON structure:
{{
  "input": {{
    "context": {{
      "score": {{...}},
      "alive_players": [...]
    }},
    "previous_events": [...],
    "current_events": [...],
    "request": {{
      "mode": "event_bundle" | "idle_color" | "idle_conversation",
      "lines": [
        {{"caster": "play_by_play" | "color", "style": "play_by_play_event" | "play_by_play_follow_up" | "idle_color"}}
      ]
    }}
  }},
  "output": {{
    "commentary": "line 1\\nline 2"
  }}
}}

Hard requirements:
- input.request must equal this exact object: {json.dumps(request, sort_keys=True)}
- output.commentary must contain exactly {expected_line_count} lines.
- Each output line must match the corresponding caster/style in input.request.lines.
- Return compact JSON only on a single line.
- Use valid JSON with double quotes.
- Escape any quotation marks inside strings.
- Do not put quotation marks inside commentary unless absolutely necessary.
'''

    response = client.responses.create(
        model=DATAGEN_MODEL,
        input=[{"role": "user", "content": prompt}],
        temperature=TEMPERATURE,
    )

    parsed = extract_json_object(response.output_text)
    return SyntheticTrainingRow.model_validate(parsed)


## Dataset generation functions


In [6]:
from tqdm import tqdm


def generate_dataset(num_examples: int, filename: str) -> None:
    if "SEED_EXAMPLES" not in globals():
        raise RuntimeError("SEED_EXAMPLES is not defined. Run the seed-loading cell first.")
    if not SEED_EXAMPLES:
        raise RuntimeError("SEED_EXAMPLES is empty. Check SEED_EXAMPLES_FILE and reload seeds.")

    with open(filename, "w", encoding="utf-8") as f:
        for idx in tqdm(range(num_examples)):
            row = None
            last_error = None
            for attempt in range(1, MAX_ROW_RETRIES + 1):
                try:
                    row = generate_synthetic_row(SEED_EXAMPLES)
                    break
                except Exception as error:
                    last_error = error
                    print(f"Error generating row {idx} on attempt {attempt}/{MAX_ROW_RETRIES}: {error}")
            if row is None:
                raise RuntimeError(f"Failed to generate row {idx} after {MAX_ROW_RETRIES} attempts") from last_error
            f.write(row.model_dump_json() + "\n")
            f.flush()



## Generate all the data!


In [ ]:
from datetime import datetime

timestamp = datetime.now().strftime('%Y-%m-%d-%H:%M:%S')
TRAIN_FILE = f"{DATA_FOLDER}/train_{timestamp}.jsonl"
VALID_FILE = f"{DATA_FOLDER}/valid_{timestamp}.jsonl"
TEST_FILE = f"{DATA_FOLDER}/test_{timestamp}.jsonl"

GENERATED_FILES = []

if NUM_TRAIN_EXAMPLES > 0:
    generate_dataset(NUM_TRAIN_EXAMPLES, TRAIN_FILE)
    GENERATED_FILES.append(("train", TRAIN_FILE))

if NUM_VAL_EXAMPLES > 0:
    generate_dataset(NUM_VAL_EXAMPLES, VALID_FILE)
    GENERATED_FILES.append(("valid", VALID_FILE))

if NUM_TEST_EXAMPLES > 0:
    generate_dataset(NUM_TEST_EXAMPLES, TEST_FILE)
    GENERATED_FILES.append(("test", TEST_FILE))

for split_name, split_path in GENERATED_FILES:
    print(f"{split_name}: {split_path}")


In [ ]:
# Preview generated rows
import json
from pathlib import Path

for split_name, split_path in GENERATED_FILES:
    preview_path = Path(split_path)
    if not preview_path.exists():
        continue
    print(f"=== {split_name} preview ===")
    for raw_line in preview_path.read_text(encoding='utf-8').splitlines()[:3]:
        print(json.dumps(json.loads(raw_line), indent=2, sort_keys=True))
        print('---')
